# P100 + Py3.10 conda env: SadTalker validation (PYTHONPATH isolation fix)

In [ ]:
# 0. Install miniconda + Py3.10 env (conda-forge) with P100-compatible torch cu117 (sm_60)
import subprocess, sys, os
MC='/kaggle/working/miniconda3'
PY=MC+'/envs/edge/bin/python'
def env_clear():
    e=dict(os.environ); e['PYTHONPATH']=''; e['PATH']=MC+'/envs/edge/bin:'+e.get('PATH',''); return e
if not os.path.exists(MC):
    subprocess.run(['bash','-c','wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /kaggle/working/mc.sh && bash /kaggle/working/mc.sh -b -p '+MC],check=True)
subprocess.run([MC+'/bin/conda','create','-y','-n','edge','python=3.10','-c','conda-forge','--override-channels'],check=True)
def cenv(cmd):
    print('$',cmd[:3])
    return subprocess.run([PY]+cmd,env=env_clear(),check=False,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL).returncode
cenv(['-m','pip','install','--upgrade','pip','-q'])
cenv(['-m','pip','install','-q','--no-cache-dir','torch==2.0.1+cu117','torchvision==0.15.2+cu117','--index-url','https://download.pytorch.org/whl/cu117'])
pkgs=['numpy==1.23.5','scipy==1.10.1','scikit-image==0.19.3','imageio==2.31.1','imageio-ffmpeg==0.4.7','pydub==0.25.1','resampy==0.3.1','joblib==1.2.0','librosa==0.9.2','numba==0.56.4','yacs==0.1.8','pyyaml','tqdm','av==10.0.1','safetensors','kornia==0.6.8','face_alignment==1.3.5','basicsr==1.4.2','facexlib==0.3.0','gfpgan','einops','opencv-python-headless','tensorboard','dlib','diffusers','transformers','accelerate','huggingface-hub']
for p in pkgs: cenv(['-m','pip','install','-q',p])
rc=cenv(['-c','import torch,numpy,skimage; print("TORCH",torch.__version__,"CUDA",torch.cuda.is_available(),"ARCH",torch.cuda.get_arch_list()); print("NUMPY",numpy.__version__,"SKIMAGE",skimage.__version__)'])
print('verify rc', rc)
print('conda env ready')


In [ ]:
# 1. Profile + 30s Kokoro voice (system python)
import subprocess, sys, os
subprocess.run([sys.executable,'-m','pip','install','-q','gdown','kokoro','soundfile'],check=False,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
subprocess.run(['apt-get','-qq','install','-y','ffmpeg'],check=False,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
import imageio_ffmpeg, shutil
FF=imageio_ffmpeg.get_ffmpeg_exe()
if not os.path.exists('/usr/local/bin/ffmpeg'): shutil.copy(FF,'/usr/local/bin/ffmpeg'); os.chmod('/usr/local/bin/ffmpeg',0o755)
os.makedirs('/kaggle/working/test',exist_ok=True)
subprocess.run(['gdown','1-2sFUEHqXDbaPq0lfBmamrjQBsdL_QuY','-O','/kaggle/working/test/profile.jpg'],check=False)
print('profile size', os.path.getsize('/kaggle/working/test/profile.jpg'))
from kokoro import KPipeline
import soundfile as sf, numpy as np
pipe=KPipeline(lang_code='a')
txt='My name is Mirsina Aghdam, CEO of EDGE, Earthwise Dynamics Geo Environs. We are an Irish company working in geoengineering, AI automation and critical-mineral intelligence. Europe needs secure rare-earth supplies, but exploration remains slow and fragmented.'
chunks=[]
for gs,ps,a in pipe(txt, voice='am_michael', speed=1.0): chunks.append(a.cpu().numpy())
a=np.concatenate(chunks) if chunks else np.zeros(24000*30)
a=a[:24000*30] if len(a)>=24000*30 else np.pad(a,(0,24000*30-len(a)))
sf.write('/kaggle/working/test/seg.wav', a.astype('float32'), 24000)
print('audio', os.path.getsize('/kaggle/working/test/seg.wav'))


In [ ]:
# 2. Clone SadTalker + download models
import os, subprocess
if not os.path.exists('/kaggle/working/SadTalker'):
    subprocess.run(['git','clone','https://github.com/OpenTalker/SadTalker.git','/kaggle/working/SadTalker'],check=True)
os.chdir('/kaggle/working/SadTalker')
r=subprocess.run(['bash','scripts/download_models.sh'],check=False)
print('models dl rc', r.returncode, 'present', os.path.exists('checkpoints/SadTalker_V0.0.2_512.safetensors'))


In [ ]:
# 3. Run SadTalker ONE clip via conda env python (PYTHONPATH cleared)
import os, subprocess, glob, shutil
MC='/kaggle/working/miniconda3'
PY=MC+'/envs/edge/bin/python'
def env_clear():
    e=dict(os.environ); e['PYTHONPATH']=''; e['PATH']=MC+'/envs/edge/bin:'+e.get('PATH',''); return e
os.chdir('/kaggle/working/SadTalker')
cmd=[PY,'inference.py','--driven_audio','/kaggle/working/test/seg.wav','--source_image','/kaggle/working/test/profile.jpg','--result_dir','/kaggle/working/sad_test','--size','512','--preprocess','crop','--still','--enhancer','gfpgan','--batch_size','1']
print('running sadtalker...'); r=subprocess.run(cmd,env=env_clear(),check=False)
print('rc', r.returncode)
files=glob.glob('/kaggle/working/sad_test/*/*.mp4')
print('outputs', files)
if files:
    shutil.copy(files[0],'/kaggle/working/test/sad_clip.mp4'); print('copied', files[0])


In [ ]:
# 4. Report
import os, glob
print('SADTALKER clip:', glob.glob('/kaggle/working/test/sad_clip.mp4'))
print('AUDIO:', os.path.exists('/kaggle/working/test/seg.wav'))
print('PROFILE:', os.path.exists('/kaggle/working/test/profile.jpg'))
